In [18]:
# ============================================================
# SMB Gold Transform
# Input : silver_smb_accounts (58,500 rows, 27 columns)
# Output: gold_smb_account_snapshot   - one row per account
#         gold_smb_monthly_kpis       - one row per month
#         gold_smb_segment_kpis       - one row per month × segment
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_TABLE   = "silver_smb_accounts"
SNAP_TABLE     = "gold_smb_account_snapshot"
MONTHLY_TABLE  = "gold_smb_monthly_kpis"
SEGMENT_TABLE  = "gold_smb_segment_kpis"

df = spark.table(SILVER_TABLE).filter(F.col("is_pre_signup") == 0)

print(f"Silver rows loaded (post-signup only): {df.count():,}")
print(f"Columns: {len(df.columns)}")

StatementMeta(, 4c8c149f-801c-4d8a-a5bd-8e4aa1157878, 20, Finished, Available, Finished, False)

Silver rows loaded (post-signup only): 39,097
Columns: 27


In [19]:
# Schema verification - confirm Silver columns are present
required_cols = [
    "seat_growth_mom", "azure_active_flag",
    "product_breadth_score", "support_spike_flag", "active_user_rate",
    "months_without_growth", "teams_adoption_pct"
]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    print(f"MISSING COLUMNS - fix Silver before proceeding: {missing}")
else:
    print("All required columns present ✓")
    print(f"Silver columns available: {len(df.columns)}")

StatementMeta(, 4c8c149f-801c-4d8a-a5bd-8e4aa1157878, 21, Finished, Available, Finished, False)

MISSING COLUMNS - fix Silver before proceeding: ['teams_adoption_pct']


In [20]:
# Base KPI engineering

# These are account-month level metrics - one value per row.
# They feed both the window calculations and the aggregations.

df = (df
  # Teams adoption % - what fraction of active M365 users use Teams daily
  # Why: Teams is Microsoft's stickiest product. Low Teams = low stickiness.
  .withColumn("teams_adoption_pct",
      F.when(F.col("m365_active_users") > 0,
             F.round(F.col("teams_daily_active") / F.col("m365_active_users"), 4))
       .otherwise(0.0))

  # Dynamics penetration % - CRM adoption depth
  .withColumn("dynamics_penetration_pct",
      F.when(F.col("m365_active_users") > 0,
             F.round(F.col("dynamics_users") / F.col("m365_active_users"), 4))
       .otherwise(0.0))

  # Azure intensity - compute hours per active user
  # Why: measures how deeply Azure is embedded, not just whether it's adopted
  .withColumn("azure_intensity",
      F.when(F.col("m365_active_users") > 0,
             F.round(F.col("azure_compute_hours") / F.col("m365_active_users"), 2))
       .otherwise(0.0))

  # Support burden rate - tickets per active user
  # Why: raw ticket count is misleading for different-sized accounts
  .withColumn("support_burden_rate",
      F.when(F.col("m365_active_users") > 0,
             F.round(F.col("support_tickets") / F.col("m365_active_users"), 4))
       .otherwise(0.0))

  # Employee band - size segmentation for dashboard slicing
  .withColumn("employee_band",
      F.when(F.col("employee_count") < 50,  "10–49")
       .when(F.col("employee_count") < 200, "50–199")
       .otherwise("200–500"))
)

print("Base KPIs engineered.")
display(df.select(
    "account_id", "month",
    "active_user_rate", "teams_adoption_pct",
    "dynamics_penetration_pct", "azure_intensity",
    "support_burden_rate", "employee_band"
).limit(10))

StatementMeta(, 4c8c149f-801c-4d8a-a5bd-8e4aa1157878, 22, Finished, Available, Finished, False)

Base KPIs engineered.


SynapseWidget(Synapse.DataFrame, 9c71df6f-a4d4-47e8-ab81-f0f024819d53)

In [21]:
# Window metrics - MoM and rolling averages


w = Window.partitionBy("account_id").orderBy("month")
w3 = w.rowsBetween(-2, 0)   # 3-month rolling window
w6 = w.rowsBetween(-5, 0)   # 6-month rolling window

df = (df
  # Previous month values for MoM deltas
  .withColumn("prev_active_users",
      F.lag("m365_active_users", 1).over(w))

  # Active user growth MoM
  .withColumn("active_users_growth_mom",
      F.when(F.col("prev_active_users").isNull(), None)
       .otherwise(F.col("m365_active_users") - F.col("prev_active_users")))

  # Usage momentum - change in active_user_rate vs last month
  .withColumn("prev_active_user_rate",
      F.lag("active_user_rate", 1).over(w))
  .withColumn("usage_momentum",
      F.when(F.col("prev_active_user_rate").isNull(), None)
       .otherwise(
           F.round(F.col("active_user_rate") - F.col("prev_active_user_rate"), 4)))

  # Rolling averages
  .withColumn("avg_active_user_rate_3mo",
      F.round(F.avg("active_user_rate").over(w3), 4))
  .withColumn("avg_active_user_rate_6mo",
      F.round(F.avg("active_user_rate").over(w6), 4))
)

print("Window metrics computed.")
display(df.select(
    "account_id", "month",
    "active_user_rate", "prev_active_user_rate",
    "usage_momentum",
    "avg_active_user_rate_3mo", "avg_active_user_rate_6mo"
).orderBy("account_id", "month").limit(15))

StatementMeta(, 4c8c149f-801c-4d8a-a5bd-8e4aa1157878, 23, Finished, Available, Finished, False)

Window metrics computed.


SynapseWidget(Synapse.DataFrame, 7efb5930-11fa-4f27-8747-f8fad9cd014c)

In [22]:
# Trend direction classification

# Compares current rate to 6-month average.
# +/-0.05 band = Stable. Outside = Improving or Declining.
# This is what appears on the Power BI account health tile.

df = df.withColumn("trend_direction",
    F.when(F.col("active_user_rate") > F.col("avg_active_user_rate_6mo") + 0.05,
           "Improving")
     .when(F.col("active_user_rate") < F.col("avg_active_user_rate_6mo") - 0.05,
           "Declining")
     .otherwise("Stable"))

print("Trend direction distribution:")
display(df.groupBy("trend_direction").count().orderBy(F.desc("count")))

StatementMeta(, 4c8c149f-801c-4d8a-a5bd-8e4aa1157878, 24, Finished, Available, Finished, False)

Trend direction distribution:


SynapseWidget(Synapse.DataFrame, 6420af9b-19b4-4397-be52-3fef3b585d22)

In [23]:
# Health score - weighted, explainable

# This is the most important Gold-layer addition.
# Every component is documented so you can explain it in an interview.
#
# Weights:
#   35 pts - active_user_rate        (core engagement)
#   20 pts - teams_adoption_pct      (collaboration depth)
#   15 pts - product_breadth_score   (ecosystem stickiness)  [0–4 → /4]
#   10 pts - seat_growth_mom         (growth momentum)
#   10 pts - azure_active_flag       (protective ecosystem signal)
#   10 pts - support_spike PENALTY   (distress signal)
#
# Total possible: 100

df = df.withColumn("health_score",
    F.round(
      F.least(F.lit(100.0), F.greatest(F.lit(0.0),
        # Engagement component (35 pts)
        F.col("active_user_rate") * 35 +

        # Teams adoption (20 pts)
        F.col("teams_adoption_pct") * 20 +

        # Product breadth (15 pts — normalize 0-4 to 0-1)
        (F.col("product_breadth_score") / 4.0) * 15 +

        # Seat growth momentum (10 pts)
        # +10 if growing, +5 if stable, 0 if shrinking
        F.when(F.col("seat_growth_mom") > 0,  10.0)
         .when(F.col("seat_growth_mom") == 0,  5.0)
         .otherwise(0.0) +

        # Azure protective signal (10 pts)
        F.col("azure_active_flag") * 10.0 +

        # Support spike PENALTY (-10 pts)
        F.when(F.col("support_spike_flag") == 1, -10.0)
         .otherwise(0.0)
      )), 1)
)

# Health label
df = df.withColumn("health_label",
    F.when(F.col("health_score") >= 80, "Healthy")
     .when(F.col("health_score") >= 60, "Stable")
     .when(F.col("health_score") >= 40, "Watchlist")
     .otherwise("At Risk"))

print("Health score distribution:")
display(df.groupBy("health_label").count().orderBy("health_label"))

display(df.select(
    "account_id", "month",
    "active_user_rate", "teams_adoption_pct",
    "product_breadth_score", "azure_active_flag",
    "support_spike_flag", "health_score", "health_label"
).limit(10))

StatementMeta(, 4c8c149f-801c-4d8a-a5bd-8e4aa1157878, 25, Finished, Available, Finished, False)

Health score distribution:


SynapseWidget(Synapse.DataFrame, 4566c63f-3e2d-46a7-92b9-4e4ec042b7f5)

SynapseWidget(Synapse.DataFrame, 49f4c0e7-3416-4d08-84b9-c7af9a2c1c8f)

In [24]:
# Business flags + recommended_action


df = (df
  .withColumn("retention_risk_flag",
      F.when(
          (F.col("health_label").isin(["At Risk", "Watchlist"])) &
          (F.col("churned") == 0), 1)
       .otherwise(0))

  .withColumn("expansion_ready_flag",
      F.when(
          (F.col("churned") == 0) &
          (F.col("active_user_rate") >= 0.75) &
          (F.col("product_breadth_score") >= 2) &
          (F.col("trend_direction").isin(["Improving", "Stable"])), 1)
       .otherwise(0))

  # FIXED: revenue_at_risk now flags AT-RISK accounts BEFORE they churn
  .withColumn("revenue_at_risk",
      F.when(
          (F.col("health_label").isin(["At Risk", "Watchlist"])) &
          (F.col("revenue_tier").isin(["High", "Mid"])),
          F.col("estimated_clv_12mo"))
       .otherwise(F.lit(0.0)))

  .withColumn("retention_priority_score",
      F.when(F.col("churned") == 0,
             F.round(
                 (1 - F.col("active_user_rate")) *
                 F.when(F.col("revenue_tier") == "High", 3.0)
                  .when(F.col("revenue_tier") == "Mid",  2.0)
                  .otherwise(1.0) *
                 F.when(F.col("health_label") == "At Risk",   1.5)
                  .when(F.col("health_label") == "Watchlist", 1.2)
                  .otherwise(1.0),
             2))
       .otherwise(0.0))
)

# Decision Engine
df = df.withColumn("recommended_action",
    F.when(
        (F.col("health_label") == "At Risk") &
        (F.col("revenue_tier") == "High"),
        "Critical Retention Call")
    .when(
        (F.col("support_spike_flag") == 1) &
        (F.col("trend_direction") == "Declining"),
        "Customer Success Intervention")
    .when(
        (F.col("health_label").isin(["At Risk","Watchlist"])) &
        (F.col("product_breadth_score") == 1),
        "Adoption Recovery — Expand Beyond Single Product")
    .when(
        (F.col("expansion_ready_flag") == 1) &
        (F.col("azure_active_flag") == 0),
        "Cross-Sell: Azure Trial Campaign")
    .when(
        (F.col("expansion_ready_flag") == 1) &
        (F.col("dynamics_users") == 0),
        "Cross-Sell: Dynamics 365 Introduction")
    .when(
        F.col("expansion_ready_flag") == 1,
        "Upsell: Copilot License Recommendation")
    .when(
        F.col("health_label") == "Healthy",
        "Nurture: Community Program")
    .otherwise("Monitor: Standard Check-in")
)

print("Action distribution:")
display(df.groupBy("recommended_action").count().orderBy(F.desc("count")))

StatementMeta(, 4c8c149f-801c-4d8a-a5bd-8e4aa1157878, 26, Finished, Available, Finished, False)

Action distribution:


SynapseWidget(Synapse.DataFrame, d8e60040-a9b1-47d8-8998-5b0bcd228451)

In [25]:
# gold_smb_account_snapshot
# Grain: one row per account - latest month snapshot

w_latest = Window.partitionBy("account_id").orderBy(F.desc("month"))

snapshot = (df
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .drop("rn", "prev_active_users", "prev_active_user_rate",
          "industry_raw", "revenue_tier_raw",
          "industry_clean", "revenue_tier_clean", "month_raw",
          "m365_active_flag", "dynamics_active_flag",
          "no_growth_flag", "growth_group", "growth_reset_flag",
          "prev_licensed_seats", "is_pre_signup")
)

snapshot.write.mode("overwrite").saveAsTable(SNAP_TABLE)

count = spark.table(SNAP_TABLE).count()
print(f"{SNAP_TABLE} saved: {count:,} rows")
print("Should equal number of distinct accounts in Silver")

# Show top 10 critical accounts
print("\nTop 10 Critical Accounts by Retention Priority:")
display(
    spark.table(SNAP_TABLE)
    .filter(F.col("recommended_action").contains("Critical"))
    .orderBy(F.desc("retention_priority_score"))
    .select("company_name", "industry", "revenue_tier",
            "health_score", "health_label", "active_user_rate",
            "product_breadth_score", "recommended_action",
            "estimated_clv_12mo")
    .limit(10)
)

StatementMeta(, 4c8c149f-801c-4d8a-a5bd-8e4aa1157878, 27, Finished, Available, Finished, False)

gold_smb_account_snapshot saved: 5,000 rows
Should equal number of distinct accounts in Silver

Top 10 Critical Accounts by Retention Priority:


SynapseWidget(Synapse.DataFrame, 0f068920-46b9-4a46-b3b9-8c36ecb0ff36)

In [26]:
# gold_smb_monthly_kpis
# Grain: one row per month - executive trend table


monthly = df.groupBy("month").agg(
    F.count("account_id").alias("total_accounts"),
    F.sum(F.when(F.col("m365_active_users") > 0, 1).otherwise(0))
     .alias("active_accounts"),
    F.sum("churned").alias("churned_accounts"),
    F.round(F.avg("active_user_rate"), 4).alias("avg_active_user_rate"),
    F.round(F.avg("teams_adoption_pct"), 4).alias("avg_teams_adoption_pct"),
    F.round(F.avg("product_breadth_score"), 3).alias("avg_product_breadth_score"),
    F.round(F.avg("health_score"), 2).alias("avg_health_score"),
    F.sum("support_spike_flag").alias("support_spike_accounts"),
    F.sum("retention_risk_flag").alias("retention_risk_accounts"),
    F.sum("expansion_ready_flag").alias("expansion_ready_accounts"),
    F.round(F.sum("revenue_at_risk"), 0).alias("estimated_revenue_at_risk")
).withColumn("monthly_churn_rate",
    F.round(F.col("churned_accounts") / F.col("total_accounts"), 4)
).orderBy("month")

monthly.write.mode("overwrite").saveAsTable(MONTHLY_TABLE)

print(f"{MONTHLY_TABLE} saved")
display(spark.table(MONTHLY_TABLE))

StatementMeta(, 4c8c149f-801c-4d8a-a5bd-8e4aa1157878, 28, Finished, Available, Finished, False)

gold_smb_monthly_kpis saved


SynapseWidget(Synapse.DataFrame, a5c62d2c-b052-4fc7-85f0-e3e0903298bc)

In [27]:
# gold_smb_segment_kpis
# Grain: one row per month × industry × region × revenue_tier × employee_band


segment = df.groupBy(
    "month", "industry", "region", "revenue_tier", "employee_band"
).agg(
    F.count("account_id").alias("accounts"),
    F.sum("churned").alias("churned_accounts"),
    F.round(F.avg("active_user_rate"), 4).alias("avg_active_user_rate"),
    F.round(F.avg("teams_adoption_pct"), 4).alias("avg_teams_adoption_pct"),
    F.round(F.avg("product_breadth_score"), 3).alias("avg_product_breadth_score"),
    F.round(F.avg("health_score"), 2).alias("avg_health_score"),
    F.round(
        F.sum("support_spike_flag") / F.count("account_id"), 4
    ).alias("support_spike_rate"),
    F.round(F.sum("revenue_at_risk"), 0).alias("estimated_revenue_at_risk")
).withColumn("churn_rate",
    F.round(F.col("churned_accounts") / F.col("accounts"), 4)
).orderBy("month", "industry", "region")

segment.write.mode("overwrite").saveAsTable(SEGMENT_TABLE)

print(f"{SEGMENT_TABLE} saved")
print(f"Total segment rows: {spark.table(SEGMENT_TABLE).count():,}")
display(spark.table(SEGMENT_TABLE).limit(15))

StatementMeta(, 4c8c149f-801c-4d8a-a5bd-8e4aa1157878, 29, Finished, Available, Finished, False)

gold_smb_segment_kpis saved
Total segment rows: 2,590


SynapseWidget(Synapse.DataFrame, a9115264-38e0-40c4-9ad2-ac20c3f89570)

In [28]:
# Gold validation


print("=" * 60)
print("GOLD LAYER VALIDATION")
print("=" * 60)

snap   = spark.table(SNAP_TABLE)
monthly = spark.table(MONTHLY_TABLE)
seg    = spark.table(SEGMENT_TABLE)

# Snapshot checks
total_snap    = snap.count()
distinct_acct = snap.select("account_id").distinct().count()

print(f"\n[{SNAP_TABLE}]")
print(f"  Rows              : {total_snap:,}")
print(f"  Distinct accounts : {distinct_acct:,}")
print(f"  One row/account   : {'YES ✓' if total_snap == distinct_acct else 'NO ✗'}")
print(f"  Health score 0–100: {snap.filter((F.col('health_score') < 0) | (F.col('health_score') > 100)).count() == 0 and '✓' or '✗'}")

# Action distribution
print(f"\n  Recommended Action breakdown:")
display(snap.groupBy("recommended_action").count().orderBy(F.desc("count")))

# Monthly checks
print(f"\n[{MONTHLY_TABLE}]")
print(f"  Months covered    : {monthly.count()}")
display(monthly.select("month","total_accounts","avg_active_user_rate",
                        "avg_health_score","estimated_revenue_at_risk"))

# Segment checks
print(f"\n[{SEGMENT_TABLE}]")
print(f"  Total segment rows: {seg.count():,}")
print(f"  Industries        : {seg.select('industry').distinct().count()}")
print(f"  Regions           : {seg.select('region').distinct().count()}")

StatementMeta(, 4c8c149f-801c-4d8a-a5bd-8e4aa1157878, 30, Finished, Available, Finished, False)

GOLD LAYER VALIDATION

[gold_smb_account_snapshot]
  Rows              : 5,000
  Distinct accounts : 5,000
  One row/account   : YES ✓
  Health score 0–100: ✓

  Recommended Action breakdown:


SynapseWidget(Synapse.DataFrame, d5cbb64c-6261-4bd3-ac1c-a71c68ac2fc9)


[gold_smb_monthly_kpis]
  Months covered    : 12


SynapseWidget(Synapse.DataFrame, bcb98e72-de15-43b6-8673-160307a42ab7)


[gold_smb_segment_kpis]
  Total segment rows: 2,590
  Industries        : 5
  Regions           : 5
